# UAE Rental Price Prediction — 04 Regression Models and Evaluation

**Goal:** predict yearly `Rent` (AED). Compare **Ridge Regression** (a regularized linear model), **Decision Tree Regression**, and a simple baseline using the same train/test split.

**Input:** `UAE_properties_features.csv` from Notebook 03.

We use 5-fold cross-validation on the training data, tune the tree with GridSearchCV, evaluate the held-out test data, and save the selected model. **`Rent_category` is excluded from inputs** because it was derived from `Rent`.

**Run the cells from top to bottom.** Calculated results appear only after you run the notebook using your dataset.

## 1. Import packages and load feature data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

paths = [
    Path('../data/UAE_properties_features.csv'),
    Path('data/UAE_properties_features.csv'),
    Path('UAE_properties_features.csv'),
    Path('/content/UAE_properties_features.csv'),
]
source_path = next((p for p in paths if p.is_file()), None)
if source_path is None:
    raise FileNotFoundError('Run Notebook 03 first, then make UAE_properties_features.csv available here.')
df = pd.read_csv(source_path, low_memory=False)
print('Loaded:', source_path)
print('Dataset size:', df.shape)
print('Missing cells:', int(df.isna().sum().sum()))
assert {'Rent','Rent_category'}.issubset(df.columns)
display(df.head(3))

## 2. Define target, features and train/test split

Hold out 20% of records for a final independent test. Do not look at test results while tuning.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

# Only property features are inputs; exclude the Rent-derived category.
X = df.drop(columns=['Rent','Rent_category']).select_dtypes(include='number').copy()
y = pd.to_numeric(df['Rent'],errors='coerce')
valid = y.notna() & (y > 0) & X.notna().all(axis=1)
X,y = X.loc[valid],y.loc[valid]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)
print('Predictors:',len(X.columns),'| Training:',len(X_train),'| Testing:',len(X_test))
print('Target:',y.name,'(AED)')

## 3. Define the models

**Baseline:** always predicts the training-set median. **Ridge:** finds a weighted linear relationship. **Decision Tree:** learns if/then rules and nonlinear relationships.

In [ ]:
models = {
    'Median baseline': DummyRegressor(strategy='median'),
    'Ridge Regression': Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('scale',StandardScaler()),
        ('model',Ridge(alpha=100.0))
    ]),
    'Decision Tree': DecisionTreeRegressor(max_depth=10,min_samples_leaf=50,random_state=42)
}

def score_model(model,X_input,y_input):
    pred = model.predict(X_input)
    return {
        'R2': r2_score(y_input,pred),
        'MAE_AED': mean_absolute_error(y_input,pred),
        'RMSE_AED': np.sqrt(mean_squared_error(y_input,pred)),
        'Median_AE_AED': median_absolute_error(y_input,pred),
    }

## 4. Cross-validation (training data only)

Each of 5 folds is used once as validation. Higher R² and lower MAE/RMSE are better. Cross-validation shows consistency between folds.

In [ ]:
cv = KFold(n_splits=5,shuffle=True,random_state=42)
scoring = {'r2':'r2','mae':'neg_mean_absolute_error','rmse':'neg_root_mean_squared_error'}
cv_rows=[]
for name,model in models.items():
    result = cross_validate(model,X_train,y_train,cv=cv,scoring=scoring,n_jobs=1)
    cv_rows.append({
        'Model':name,
        'CV_R2_mean':result['test_r2'].mean(),
        'CV_R2_std':result['test_r2'].std(),
        'CV_MAE_AED':-result['test_mae'].mean(),
        'CV_RMSE_AED':-result['test_rmse'].mean()
    })
cv_table = pd.DataFrame(cv_rows).sort_values('CV_MAE_AED')
display(cv_table.round(3))

## 5. Tune the Decision Tree

GridSearchCV tests a small list of tree settings using only training data. This helps reduce overfitting and selects the combination with the lowest validation MAE.

In [ ]:
search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid={'max_depth':[6,10],'min_samples_leaf':[50,150]},
    cv=3,scoring='neg_mean_absolute_error',n_jobs=1,refit=True
)
search.fit(X_train,y_train)
print('Best tree settings:',search.best_params_)
print('Best 3-fold CV MAE (AED):',round(-search.best_score_,2))
models['Tuned Decision Tree'] = search.best_estimator_

## 6. Final comparison on test data

The test data was not used to train or tune. **R²** measures fit (higher is better), while **MAE** and **RMSE** are errors in AED (lower is better). Outliers can make RMSE large.

In [ ]:
test_rows=[]
for name,model in models.items():
    model.fit(X_train,y_train)
    test_rows.append({'Model':name,**score_model(model,X_test,y_test)})
results = pd.DataFrame(test_rows).sort_values('MAE_AED')
display(results.round(3))
print('Note: Extreme rents may strongly increase RMSE even when typical errors are smaller.')

## 7. Plot actual vs predicted for the strongest tested model

This plot is for **visual inspection** only. We use the model with lowest reported test MAE for display; test-set model selection is exploratory, not an unbiased final model-selection method. For a deployment decision, select by training CV and then inspect its held-out test result.

In [ ]:
import matplotlib.pyplot as plt
# Select by CV, not the test set (baseline may legitimately win).
winner_name = cv_table.iloc[0]['Model']
if winner_name == 'Decision Tree':
    # Keep the tuned tree only if its 3-fold CV estimate improves the original tree's CV MAE.
    original_mae = cv_table.loc[cv_table['Model']=='Decision Tree','CV_MAE_AED'].iloc[0]
    if -search.best_score_ < original_mae:
        winner_name='Tuned Decision Tree'
best_model = models[winner_name]
best_model.fit(X_train,y_train)
predictions = best_model.predict(X_test)
plot_cap = np.quantile(y_test,0.99)
show = (y_test <= plot_cap) & (predictions <= plot_cap)
plt.figure(figsize=(7,5))
plt.scatter(y_test[show],predictions[show],s=10,alpha=0.3)
plt.plot([0,plot_cap],[0,plot_cap],linestyle='--')
plt.xlabel('Actual Rent (AED)')
plt.ylabel('Predicted Rent (AED)')
plt.title('Actual vs Predicted Rent (99th percentile view)')
plt.tight_layout()
plt.show()
print('Model selected using training CV:',winner_name)
print('Final test metrics:',score_model(best_model,X_test,y_test))

## 8. Reliability and business decision

Interpret the actual results above. A **GO** decision needs business-approved error limits and reliable performance across folds. This notebook cannot invent those limits.

In [ ]:
cv_winner = cv_table.loc[cv_table['Model'].eq('Decision Tree' if winner_name=='Tuned Decision Tree' else winner_name)].iloc[0]
print('Training CV R2 mean:',round(cv_winner['CV_R2_mean'],3))
print('Training CV R2 standard deviation:',round(cv_winner['CV_R2_std'],3))
print('Final test MAE (AED):',round(mean_absolute_error(y_test,predictions),2))
print('Business decision: NO-GO for automatic rental decisions until the team agrees on an acceptable AED error,')
print('checks high-rent errors, confirms stability, and tests fresh listings.')

## 9. Save the selected model

The saved file is useful for future Streamlit integration. The model must receive the **same feature columns** and preprocessing as in this notebook.

In [ ]:
import joblib
project_root = source_path.parent.parent if source_path.parent.name=='data' else source_path.parent
model_dir = project_root / 'models'
model_dir.mkdir(parents=True,exist_ok=True)
model_path = model_dir / 'regression_model.joblib'
joblib.dump({'model':best_model,'feature_columns':X.columns.tolist(),'target':'Rent','model_name':winner_name},model_path)
print('Saved:',model_path)
print('Input features:',len(X.columns))